# Largest Rectangle in Histogram

# Problem Statement

Given an array of integers `heights` where each element represents the height of a histogram bar, find the area of the largest rectangle that can be formed inside the histogram.

Each bar has a width of `1`.

### Input

An array of non-negative integers:

```text
heights
```

### Output

Return the maximum possible rectangle area.

### Examples

```text
Input:
[2,1,5,6,2,3]

Output:
10
```

The largest rectangle uses:

```text
5 and 6
```

The limiting height is:

```text
5
```

and the width is:

```text
2
```

Therefore:

```text
Area = 5 × 2 = 10
```

---

```text
Input:
[2,4]

Output:
4
```

The largest rectangle has:

```text
height = 2
width = 2
```

Therefore:

```text
Area = 2 × 2 = 4
```

# Problem Explanation

For every bar, imagine that the bar is the **shortest bar of a rectangle**.

If a bar has height:

```text
h
```

then the maximum rectangle using that height depends on how far we can extend:

```text
← left
right →
```

before encountering a bar shorter than `h`.

Therefore, for every bar we need to know:

```text
Previous Smaller Element
```

and:

```text
Next Smaller Element
```

Suppose:

```text
[2, 1, 5, 6, 2, 3]
```

For the bar with height `5`:

```text
Previous smaller = 1
Next smaller = 2
```

So the rectangle can extend across:

```text
5, 6
```

giving:

```text
height = 5
width = 2
area = 10
```

This gives us the key idea:

```text
Largest Rectangle
       ↓
Previous Smaller + Next Smaller
       ↓
Monotonic Stack
```

# Brute Force

For every bar, consider it as the minimum-height bar.

Then expand:

```text
left
```

and:

```text
right
```

until encountering a smaller bar.

For example:

```text
[2,1,5,6,2,3]
```

For height `5`:

```text
left boundary → 1
right boundary → 2
```

So:

```text
width = 2
area = 5 × 2 = 10
```

Doing this for every bar requires repeatedly scanning left and right.

### Complexity

```text
Time → O(N²)
Space → O(1)
```

We can improve this using a Monotonic Stack.

# The Key Idea

For each bar:

```text
height = heights[i]
```

we want the largest width over which this height can be the minimum.

That means finding:

```text
Previous Smaller
```

and:

```text
Next Smaller
```

If:

```text
left = index of previous smaller
right = index of next smaller
```

then the usable width is:

```text
right - left - 1
```

Therefore:

```text
area = height × width
```

The entire problem becomes:

```text
Find boundaries efficiently
            ↓
Monotonic Stack
```

# Why a Monotonic Increasing Stack?

We maintain indices whose heights are increasing:

```text
1
2
5
6
```

When we encounter a smaller height:

```text
2
```

we know that every taller bar on the Stack has just found its:

```text
Next Smaller Element
```

For example:

```text
[1, 5, 6, 2]
```

When `2` arrives:

```text
6 > 2
```

So `6` can no longer extend to the right.

Then:

```text
5 > 2
```

So `5` can no longer extend to the right either.

This allows us to calculate their rectangle areas immediately.

# The Width Calculation

Suppose the Stack contains:

```text
[1, 5, 6]
```

and we encounter:

```text
2
```

For height `6`:

```text
next smaller = current index
previous smaller = index of 5
```

Therefore:

```text
width = current_index - previous_smaller_index - 1
```

For height `5`, after removing `6`:

```text
previous smaller = index of 1
next smaller = current index
```

Therefore:

```text
width = current_index - previous_smaller_index - 1
```

The `-1` is important because the boundary bars themselves cannot be included.

# Algorithm

Add a virtual bar of height `0` at the end.

This forces every remaining bar in the Stack to be processed.

For every index:

```text
While Stack is not empty
and current height < height at Stack top:

    height = height of popped bar

    right boundary = current index

    If Stack is empty:
        left boundary = -1
    Else:
        left boundary = Stack top

    width = right boundary - left boundary - 1

    area = height × width

    update maximum area

Push current index
```

The final zero acts as a cleanup mechanism.

In [1]:
class Solution:

    def largestRectangleArea(self, heights: list[int]) -> int:

        stack = []
        max_area = 0

        heights.append(0)

        for i in range(len(heights)):

            while stack and heights[i] < heights[stack[-1]]:

                height = heights[stack.pop()]

                if stack:
                    left = stack[-1]
                else:
                    left = -1

                width = i - left - 1

                area = height * width

                max_area = max(max_area, area)

            stack.append(i)

        heights.pop()

        return max_area

# Dry Run

Input:

```text
[2,1,5,6,2,3]
```

Add the virtual zero:

```text
[2,1,5,6,2,3,0]
```

Start:

```text
Stack = []
Max Area = 0
```

### Index 0 — Height 2

Push:

```text
Stack = [0]
```

---

### Index 1 — Height 1

Current:

```text
1 < 2
```

Pop index `0`.

```text
height = 2
```

Stack is now empty:

```text
left = -1
```

Current index:

```text
right = 1
```

Width:

```text
1 - (-1) - 1 = 1
```

Area:

```text
2 × 1 = 2
```

Maximum:

```text
2
```

Push index `1`:

```text
Stack = [1]
```

---

### Index 2 — Height 5

```text
5 > 1
```

Push:

```text
Stack = [1,2]
```

---

### Index 3 — Height 6

```text
6 > 5
```

Push:

```text
Stack = [1,2,3]
```

---

### Index 4 — Height 2

Current:

```text
2 < 6
```

Pop `6`.

```text
height = 6
left = 2
right = 4
```

Width:

```text
4 - 2 - 1 = 1
```

Area:

```text
6 × 1 = 6
```

Maximum remains:

```text
6
```

Now compare with height `5`:

```text
2 < 5
```

Pop `5`.

```text
height = 5
left = 1
right = 4
```

Width:

```text
4 - 1 - 1 = 2
```

Area:

```text
5 × 2 = 10
```

Maximum:

```text
10
```

Now:

```text
Stack = [1]
```

Push index `4`:

```text
Stack = [1,4]
```

---

### Index 5 — Height 3

```text
3 > 2
```

Push:

```text
Stack = [1,4,5]
```

---

### Index 6 — Height 0

This is the virtual bar.

Current:

```text
0 < 3
```

Pop `3`.

```text
height = 3
left = 4
right = 6
```

Width:

```text
6 - 4 - 1 = 1
```

Area:

```text
3
```

Now:

```text
0 < 2
```

Pop `2`.

```text
height = 2
left = 1
right = 6
```

Width:

```text
6 - 1 - 1 = 4
```

Area:

```text
2 × 4 = 8
```

Maximum remains:

```text
10
```

The remaining bars are shorter than the virtual zero only if none; after the cleanup, the Stack is emptied through the same process.

Final answer:

```text
10
```

# Visual Understanding

Consider:

```text
[2, 1, 5, 6, 2, 3]
```

The important rectangle is:

```text
    █
  █ █
  █ █
  █ █
```

using:

```text
5, 6
```

The height is limited by:

```text
5
```

and the width is:

```text
2
```

Therefore:

```text
Area = 5 × 2 = 10
```

The Monotonic Stack finds the exact boundaries where height `5` can no longer extend.

```text
1 | 5 | 6 | 2
  ↑       ↑
left    right
```

So:

```text
width = 4 - 1 - 1
      = 2
```

and:

```text
area = 5 × 2
     = 10
```

# Why the Virtual Zero?

Suppose:

```text
[2,4,5]
```

There is no smaller element after `5`.

Therefore, without additional handling, `5` would remain inside the Stack.

The virtual zero:

```text
[2,4,5,0]
```

acts as a smaller element and forces all remaining bars to be processed.

So the zero is not part of the original histogram.

It is simply a convenient way to say:

```text
The histogram ends here.
```

# Edge Cases

### Single Bar

```text
Input:
[5]

Output:
5
```

---

### All Increasing

```text
Input:
[1,2,3,4,5]
```

The largest rectangle is:

```text
3 × 3 = 9
```

using:

```text
[3,4,5]
```

---

### All Equal

```text
Input:
[5,5,5]
```

Largest rectangle:

```text
5 × 3 = 15
```

---

### All Decreasing

```text
Input:
[5,4,3,2,1]
```

The algorithm correctly considers increasingly wider rectangles.

---

### Zero Heights

```text
Input:
[2,0,2]
```

The zero separates the two histogram sections.

The maximum area is:

```text
2
```

# Common Mistakes

### Mistake 1 — Getting the Width Wrong

The correct formula is:

```python
width = right - left - 1
```

Not:

```python
right - left
```

The left and right boundary bars are smaller than the rectangle and therefore cannot be included.

---

### Mistake 2 — Using the Current Height for the Area

When a bar is popped:

```python
height = heights[stack.pop()]
```

The rectangle height is the **popped bar's height**, not the current smaller height.

---

### Mistake 3 — Forgetting the Empty Stack Case

If the Stack becomes empty:

```text
left = -1
```

This means the rectangle extends all the way to the beginning.

---

### Mistake 4 — Forgetting the Cleanup

If the array is increasing:

```text
[1,2,3,4]
```

many bars remain in the Stack after the loop.

The virtual zero forces them to be processed.

---

### Mistake 5 — Modifying the Input Permanently

The implementation temporarily appends:

```text
0
```

and removes it afterward.

This keeps the original `heights` array unchanged.

# Complexity

Let:

```text
N = len(heights)
```

Every bar is:

```text
Pushed once
Popped at most once
```

Therefore:

```text
Time → O(N)
```

The Stack can contain up to `N` indices:

```text
Space → O(N)
```

The virtual zero does not change the asymptotic complexity.

# Why Is It O(N)?

The nested `while` loop looks expensive.

But just like the other Monotonic Stack problems, each index can be popped only once.

For every bar:

```text
Push → once
Pop  → at most once
```

Therefore the total number of Stack operations is proportional to `N`.

So:

```text
Time = O(N)
```

This is an example of **amortized O(1) work per element**.

# Comparison

| Approach | Time | Space |
|---|---:|---:|
| Brute Force | O(N²) | O(1) |
| Monotonic Stack | O(N) | O(N) |

The Monotonic Stack is substantially better for large histograms.

# Pattern Recognition

This problem is a major Monotonic Stack pattern.

The trigger is:

```text
Largest rectangle
+
Histogram
```

Think:

```text
For every bar:
    How far can this height extend?
```

That requires:

```text
Previous Smaller
+
Next Smaller
```

Therefore:

```text
Largest Rectangle in Histogram
          ↓
      Monotonic Stack
          ↓
Previous Smaller + Next Smaller
          ↓
       Calculate Area
```

This is one of the most important Stack patterns to recognize in interviews.

# Takeaway

The core idea is:

```text
Every bar can be the minimum height
of a rectangle.
```

For each bar, find how far it can extend before encountering a smaller bar.

```text
Previous Smaller
       ↓
    Height
       ↓
 Next Smaller
       ↓
     Width
       ↓
     Area
```

The Monotonic Stack finds these boundaries in linear time.

The key formula is:

```text
width = right - left - 1

area = height × width
```

Complexity:

```text
Time  → O(N)
Space → O(N)
```

This problem is especially important because the same boundary-based idea appears in several harder problems, including **Maximal Rectangle**.